# Notebook 12.8: Adapting a pretrained transformer: features, full fine-tuning, and LoRA

In notebooks 12.5 and 12.6 we trained sentiment classifiers for IMDB movie reviews from scratch. In practice, we almost always start from a transformer that has been *pretrained* on a large amount of text, and adapt it to our task. This notebook compares three ways of doing this with the same pretrained encoder, DistilBERT (a smaller, 6-layer version of BERT, with about 66 million parameters; see section 12.6 of the book for BERT):

* **(a) Frozen features.** Keep the pretrained model fixed, use it to compute one feature vector per review, and train a logistic regression classifier on these features.
* **(b) Full fine-tuning.** Add a classification head and update *all* the weights of the model with gradient descent.
* **(c) LoRA (low-rank adaptation).** Keep the pretrained weights frozen, and learn a small low-rank update for some of the weight matrices.

For each approach we print the number of trainable parameters, the test accuracy, and the time it takes. To keep the notebook fast, we use only 2,000 training reviews and 1,000 test reviews.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen before you run the code.

This notebook needs internet access (to download the dataset and the pretrained model) and runs in a few minutes on a GPU. In Colab, choose Runtime > Change runtime type > GPU.

In [1]:
# Run this if you're in Colab
!pip install -q peft

In [2]:
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from sklearn.linear_model import LogisticRegression

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAME = "distilbert-base-uncased"
N_TRAIN, N_TEST = 2000, 1000
if device.type == "cpu":
    # Without a GPU, use a smaller subset so that the notebook finishes in a reasonable time
    N_TRAIN, N_TEST = 500, 250
    print("No GPU found: using", N_TRAIN, "training and", N_TEST, "test reviews")
MAX_LENGTH = 256
BATCH_SIZE = 16
SEED = 0


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_all(model):
    return sum(p.numel() for p in model.parameters())


def synchronize():
    # Wait for the GPU to finish, so that the timings are correct
    if device.type == "cuda":
        torch.cuda.synchronize()

Device: cuda


## Data

We load the IMDB dataset (labels: 0 = negative, 1 = positive) and draw a random subset of the training and test splits with a fixed seed. Then we tokenize the reviews with DistilBERT's own tokenizer, a WordPiece tokenizer with a vocabulary of about 30,000 sub-word tokens (compare with the byte pair encoding of notebook 12.3). Every review is truncated or padded to `MAX_LENGTH` = 256 tokens, and the tokenizer also returns an *attention mask* that is 1 for real tokens and 0 for padding, so that the model can ignore the padding.

Note: many IMDB reviews are longer than 256 tokens, so the models only see the beginning of those reviews.

In [3]:
ds = load_dataset("imdb")
train_data = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
test_data = ds["test"].shuffle(seed=SEED).select(range(N_TEST))
print(f"Training reviews: {len(train_data)} ({sum(train_data['label'])} positive)")
print(f"Test reviews:     {len(test_data)} ({sum(test_data['label'])} positive)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print(tokenizer.tokenize("This movie was unbelievably good!"))


def make_dataset(data):
    enc = tokenizer(list(data["text"]), truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
    return TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(list(data["label"])))


train_dataset = make_dataset(train_data)
test_dataset = make_dataset(test_data)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
n_truncated = sum(len(ids) > MAX_LENGTH for ids in tokenizer(list(train_data["text"]), verbose=False)["input_ids"])
print(f"{n_truncated / len(train_data) * 100:.0f}% of the training reviews are longer than {MAX_LENGTH} tokens and are truncated")

Training reviews: 2000 (985 positive)
Test reviews:     1000 (493 positive)


Vocabulary size: 30522
['this', 'movie', 'was', 'un', '##bel', '##ie', '##va', '##bly', 'good', '!']


44% of the training reviews are longer than 256 tokens and are truncated


## (a) Frozen features and logistic regression

The pretrained DistilBERT model (without any classification head) maps a review to a sequence of 768-dimensional vectors, one per token (the output of its last transformer layer). We turn this into a single vector per review by averaging the vectors of the real tokens (mean pooling, using the attention mask to ignore the padding). DistilBERT is never updated: we run it once over all the reviews inside `torch.no_grad()`, and then train a logistic regression classifier (from scikit-learn) on the 768-dimensional features. The only trainable parameters are the 768 weights and the bias of the logistic regression.

TODO: Before running the next cell, predict the test accuracy of this approach. DistilBERT was pretrained to fill in masked words, not to classify sentiment. Will its features be good enough for a linear classifier?

In [4]:
torch.manual_seed(SEED)
start = time.time()
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
encoder.eval()


def extract_features(loader):
    features, labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, label in loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            hidden = encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (batch, tokens, 768)
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # Mean over the real tokens
            features.append(pooled.cpu().numpy())
            labels.append(label.numpy())
    return np.concatenate(features), np.concatenate(labels)


X_train, y_train = extract_features(DataLoader(train_dataset, batch_size=64, shuffle=False))
X_test, y_test = extract_features(test_loader)
print("Feature matrix:", X_train.shape)

classifier = LogisticRegression(max_iter=2000)
classifier.fit(X_train, y_train)
acc_features = classifier.score(X_test, y_test)
synchronize()
time_features = time.time() - start

params_features = classifier.coef_.size + classifier.intercept_.size
print(f"DistilBERT parameters (all frozen): {count_all(encoder):,}")
print(f"Trainable parameters (logistic regression): {params_features:,}")
print(f"Test accuracy: {acc_features*100:.2f}%")
print(f"Time (feature extraction + logistic regression): {time_features:.1f} s")

del encoder
if device.type == "cuda":
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11389.52it/s]


DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Feature matrix: (2000, 768)


DistilBERT parameters (all frozen): 66,362,880
Trainable parameters (logistic regression): 769
Test accuracy: 84.90%
Time (feature extraction + logistic regression): 25.8 s


## Training and evaluation loops

For approaches (b) and (c) we need an ordinary PyTorch training loop. `AutoModelForSequenceClassification` adds a classification head on top of DistilBERT (a hidden layer and a linear layer that act on the output vector of the first token, `[CLS]`), and when we pass the labels it also computes the cross-entropy loss for us (`outputs.loss`). The evaluation loop runs inside `torch.no_grad()` and counts the correct predictions. We evaluate on the test set only once, after training.

In [5]:
def train_one_epoch(model, loader, optimizer):
    model.train()  # Enables dropout
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()  # Disables dropout
    correct = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            correct += (logits.argmax(dim=-1) == labels).sum().item()
    return correct / len(loader.dataset)


def train_and_evaluate(model, optimizer, n_epochs):
    start = time.time()
    for epoch in range(n_epochs):
        loss = train_one_epoch(model, train_loader, optimizer)
        print(f"Epoch {epoch+1}: training loss {loss:.4f}")
    accuracy = evaluate(model, test_loader)
    synchronize()
    return accuracy, time.time() - start

## (b) Full fine-tuning

Now we update every weight of DistilBERT, together with the new classification head, for 2 epochs. We use AdamW with a small learning rate ($2\times 10^{-5}$), which is typical for fine-tuning: a large learning rate would quickly destroy the pretrained features.

TODO: Before running the next cell, predict how many parameters will be trained, and whether the test accuracy will be higher or lower than for the frozen features.

In [6]:
torch.manual_seed(SEED)
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
params_full = count_trainable(model_full)
print(f"Trainable parameters: {params_full:,} of {count_all(model_full):,}")

optimizer = torch.optim.AdamW(model_full.parameters(), lr=2e-5)
acc_full, time_full = train_and_evaluate(model_full, optimizer, n_epochs=2)
print(f"Test accuracy: {acc_full*100:.2f}%")
print(f"Time (fine-tuning + evaluation): {time_full:.1f} s")

del model_full, optimizer
if device.type == "cuda":
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2261.20it/s]


DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 66,955,010 of 66,955,010


Epoch 1: training loss 0.4354


Epoch 2: training loss 0.2123


Test accuracy: 88.40%
Time (fine-tuning + evaluation): 102.4 s


## (c) LoRA: low-rank adaptation

Full fine-tuning works well, but it produces a complete new copy of the model for every task, and the optimizer has to store extra state (the Adam moments) for every weight. LoRA (Hu et al., 2021, https://arxiv.org/abs/2106.09685) is based on the observation that the *change* in the weights needed to adapt a pretrained model to a new task can be approximated by a matrix of low rank.

For a pretrained weight matrix $\mathbf{W}\in\mathbb{R}^{d\times k}$, LoRA keeps $\mathbf{W}$ frozen and learns an update

$$\mathbf{W}' = \mathbf{W} + \Delta\mathbf{W} = \mathbf{W} + \frac{\alpha}{r}\,\mathbf{B}\mathbf{A},\qquad \mathbf{B}\in\mathbb{R}^{d\times r},\quad \mathbf{A}\in\mathbb{R}^{r\times k},$$

where the rank $r$ is much smaller than $d$ and $k$. Only $\mathbf{A}$ and $\mathbf{B}$ are trained. $\mathbf{B}$ is initialized to zero, so at the start of training $\Delta\mathbf{W} = \mathbf{0}$ and the model is exactly the pretrained model. The constant $\alpha/r$ scales the update. After training, $\Delta\mathbf{W}$ can be added to $\mathbf{W}$, so the adapted model is no slower than the original.

**Example from the lecture.** A $25\times 50$ weight matrix has $25\times 50 = 1{,}250$ entries. With rank $r=5$, the factors $\mathbf{B}$ ($25\times 5$) and $\mathbf{A}$ ($5\times 50$) have only $125 + 250 = 375$ entries.

In DistilBERT, we apply LoRA to the query and value projections of the self-attention (the layers called `q_lin` and `v_lin`), which are $768\times 768$ matrices in each of the 6 transformer layers. The cell below counts the parameters for rank $r=8$.

In [7]:
# Lecture example: a 25 x 50 weight matrix with rank r = 5
d, k, r = 25, 50, 5
print(f"Full {d} x {k} matrix: {d*k:,} entries; LoRA factors with r = {r}: {d*r:,} + {r*k:,} = {d*r + r*k:,}")

# DistilBERT: q_lin and v_lin (768 x 768) in each of the 6 layers, rank r = 8
d = k = 768
n_layers, n_matrices, r = 6, 2, 8
full_update = n_layers * n_matrices * d * k
lora_update = n_layers * n_matrices * (d * r + r * k)
print(f"Updating q_lin and v_lin directly:  {full_update:,} weights")
print(f"LoRA factors with r = {r}:            {lora_update:,} weights ({lora_update / full_update * 100:.2f}% of the full update)")

Full 25 x 50 matrix: 1,250 entries; LoRA factors with r = 5: 125 + 250 = 375
Updating q_lin and v_lin directly:  7,077,888 weights
LoRA factors with r = 8:            147,456 weights (2.08% of the full update)


We use the `peft` library from Hugging Face, which wraps the model and inserts the LoRA factors. The configuration below uses rank $r=8$, $\alpha=16$, and dropout 0.1 on the input of the LoRA branch, applied to `q_lin` and `v_lin`. With `task_type="SEQ_CLS"`, `peft` also keeps the new classification head trainable (it is randomly initialized, so it must be trained), so the total number of trainable parameters is a little larger than the count above. Because far fewer parameters are trained and the pretrained weights cannot be damaged, LoRA usually uses a larger learning rate than full fine-tuning; here we use $5\times 10^{-4}$.

TODO: Before running the next cell, predict (1) the percentage of the model's parameters that will be trainable, and (2) whether LoRA will be much faster than full fine-tuning. (Hint: the forward and backward passes still go through the whole network.)

In [8]:
torch.manual_seed(SEED)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_lin", "v_lin"], task_type="SEQ_CLS")
model_lora = get_peft_model(base_model, lora_config).to(device)
model_lora.print_trainable_parameters()
params_lora = count_trainable(model_lora)

# Look at one of the adapted layers: the frozen 768 x 768 layer plus the LoRA factors A (768 -> 8) and B (8 -> 768)
print(model_lora.base_model.model.distilbert.transformer.layer[0].attention.q_lin)

optimizer = torch.optim.AdamW([p for p in model_lora.parameters() if p.requires_grad], lr=5e-4)
acc_lora, time_lora = train_and_evaluate(model_lora, optimizer, n_epochs=2)
print(f"Test accuracy: {acc_lora*100:.2f}%")
print(f"Time (fine-tuning + evaluation): {time_lora:.1f} s")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8588.20it/s]


DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925
lora.Linear(
  (base_layer): Linear(in_features=768, out_features=768, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=768, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=768, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


Epoch 1: training loss 0.4280


Epoch 2: training loss 0.2765


Test accuracy: 87.60%
Time (fine-tuning + evaluation): 80.8 s


## Results

In [9]:
print(f"{'Approach':38s} {'Trainable parameters':>21s} {'Test accuracy':>14s} {'Time':>8s}")
print(f"{'(a) Frozen features + logistic regr.':38s} {params_features:>21,} {acc_features*100:>13.2f}% {time_features:>7.1f}s")
print(f"{'(b) Full fine-tuning':38s} {params_full:>21,} {acc_full*100:>13.2f}% {time_full:>7.1f}s")
print(f"{'(c) LoRA (r = 8, q_lin and v_lin)':38s} {params_lora:>21,} {acc_lora*100:>13.2f}% {time_lora:>7.1f}s")

Approach                                Trainable parameters  Test accuracy     Time
(a) Frozen features + logistic regr.                     769         84.90%    25.8s
(b) Full fine-tuning                              66,955,010         88.40%   102.4s
(c) LoRA (r = 8, q_lin and v_lin)                    739,586         87.60%    80.8s


| Approach | Trainable parameters | Test accuracy | Time (our GPU) |
|---|---:|---:|---:|
| (a) Frozen features + logistic regression | 769 | 84.90% | 26 s |
| (b) Full fine-tuning, 2 epochs | 66,955,010 | 88.40% | 102 s |
| (c) LoRA, $r=8$ on `q_lin` and `v_lin`, 2 epochs | 739,586 (1.09%) | 87.60% | 81 s |

(Numbers from our run with 2,000 training and 1,000 test reviews; they will vary a little with the hardware and library versions. With 1,000 test reviews, one review is worth 0.1 percentage points, and differences of about one percentage point between runs are normal.)

### Discussion

* **Frozen features are surprisingly good.** A logistic regression with only 769 parameters, trained on the features of a model that has never seen a sentiment label, reaches about 85%. That is as good as the LSTM and the transformer that we trained from scratch in notebooks 12.5 and 12.6, which used more than ten times as many training reviews (22,500). Pretraining on a large amount of text has already produced features in which positive and negative reviews are nearly linearly separable.
* **Fine-tuning helps further.** Updating the weights of the network adapts the features to the task and gains a few more percentage points. Full fine-tuning trains all 67 million parameters.
* **LoRA gets most of the benefit with about 1% of the parameters.** LoRA reaches almost the same accuracy as full fine-tuning while training 739,586 parameters: 147,456 in the LoRA factors (as computed above) and 592,130 in the new classification head (a $768\times 768$ hidden layer and a $768\times 2$ output layer, which have to be trained in every approach except (a)). The adapter for a new task can be stored in about 3 MB (in 32-bit floating point), instead of a full 270 MB copy of the model.
* **LoRA is not dramatically faster here.** The forward pass and most of the backward pass still go through the whole network, because the gradient must flow back through the frozen layers to reach the LoRA factors in the lower layers. The savings come from not computing gradients for the frozen weights and from the much smaller optimizer state. For a model with billions of parameters, the memory saved (no gradients or Adam moments for the frozen weights) is what makes fine-tuning possible on a single GPU at all.

TODO: Try some experiments:
* Change the rank $r$ (for example 1, 2, 16, or 64). How do the number of trainable parameters and the test accuracy change? How small can $r$ be before the accuracy drops?
* Apply LoRA to more layers, for example `target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"]`.
* Use more training reviews (for example 5,000). Which of the three approaches benefits most from more data?